In [2]:
import json
from pathlib import Path
from lkb.kb.uriel_plus import URIELPlus
from lkb.eval.splits import Split
from lkb.eval.evaluator import Evaluator

In [3]:
kb = URIELPlus.from_artifacts(
    ".",
    typology_path="../data/derived/URIELPlus_Union.csv",
)

print(f"Languages: {len(kb.languages):,}")
print(f"Features:  {len(kb.features):,}")

obs = kb.observed_mask()
print(f"Observed entries: {obs.sum():,}  ({obs.mean()*100:.1f}% of matrix)")

Languages: 8,171
Features:  800
Observed entries: 888,741  (13.6% of matrix)


In [4]:
split = Split.stratified(
    kb,
    n_per_cell=100,
    lrl_frac=0.33,
    mrl_frac=0.34,
    min_observed=1,
    include_dev=True,
    seed=42,
)

split.save("../../data/splits/splits_v2.json")
print(f"Test pairs: {len(split.test):,}")
print(f"Dev  pairs: {len(split.dev):,}")

print()
print("Cell summary (from meta):")
for cell, stats in split.meta["cells"].items():
    print(f"  {cell:<20}  available={stats['available']:>6,}  test={stats['test']}  dev={stats['dev']}")

Test pairs: 1,200
Dev  pairs: 1,200

Cell summary (from meta):
  hrl/INV               available=209,230  test=100  dev=100
  hrl/M                 available=80,051  test=100  dev=100
  hrl/P                 available=29,960  test=100  dev=100
  hrl/S                 available=191,264  test=100  dev=100
  lrl/INV               available= 6,268  test=100  dev=100
  lrl/M                 available=27,192  test=100  dev=100
  lrl/P                 available= 2,584  test=100  dev=100
  lrl/S                 available=62,612  test=100  dev=100
  mrl/INV               available=117,110  test=100  dev=100
  mrl/M                 available=50,845  test=100  dev=100
  mrl/P                 available=16,603  test=100  dev=100
  mrl/S                 available=95,022  test=100  dev=100


In [5]:
gold_items = Evaluator.gold_items_from_split(kb, split, partition="test")

GOLD_OUT = Path("../../data/benchmark/gold_eval_2.jsonl")
GOLD_OUT.parent.mkdir(parents=True, exist_ok=True)
with GOLD_OUT.open("w", encoding="utf-8") as f:
    for item in gold_items:
        record = {
            "id": item.id,
            "resource_group": item.resource_group,
            "language": item.language,
            "feature": item.feature,
            "gold_value": item.gold_value,
            "allowed_values": list(item.allowed_values),
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"{len(gold_items)} items")

1200 items
